In [1]:
import pandas as pd
from scipy import stats

df = pd.read_csv("da_sales_transactions.csv")
df.head()

,Transaction_ID,Date,Customer_ID,Product_ID,Category,Units,Unit_Price,Discount_Pct,Channel,City,Return_Flag
0,T1,2025-01-24,C2,P2,Sports,5,539,15,Marketplace,Pune,1
1,T2,2025-12-15,C3,P3,Grocery,1,4228,5,Marketplace,Delhi,0
2,T3,2025-10-11,C4,P4,Books,2,2061,15,Offline,Chennai,0
3,T4,2025-04-24,C5,P5,Electronics,3,2164,0,Website,Chennai,0
4,T5,2025-06-25,C6,P6,Apparel,1,2341,0,Website,Kolkata,0


# Scenario 1 — One-sample t-test

Business question: Is the average unit price different from ₹500?

H₀: Mean Unit Price = ₹500
H₁: Mean Unit Price ≠ ₹500
Significance level: α = 0.05

In [3]:
sample = df["Unit_Price"]

t_stat, p_value = stats.ttest_1samp(sample, 500)

print("T-statistic:", t_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject H0: The average Unit Price is significantly different from ₹500.")
else:
    print("Fail to reject H0: There is not enough evidence that the average Unit Price differs from ₹500.")

T-statistic: 54.22639195906186
P-value: 6.690422313764842e-300
Reject H0: The average Unit Price is significantly different from ₹500.


# Scenario 2 — Two-sample t-test

Business question: Is average Unit Price different between Online and Offline sales?

H₀: Mean Online Unit Price = Mean Offline Unit Price
H₁: Mean Online Unit Price ≠ Mean Offline Unit Price

In [4]:
online = df[df["Channel"] == "Online"]["Unit_Price"]
offline = df[df["Channel"] == "Offline"]["Unit_Price"]

t_stat, p_value = stats.ttest_ind(online, offline, equal_var=False)

print("T-statistic:", t_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject H0: Average Unit Price differs significantly between Online and Offline sales.")
else:
    print("Fail to reject H0: No significant difference was found between Online and Offline Unit Price.")

T-statistic: nan
P-value: nan
Fail to reject H0: No significant difference was found between Online and Offline Unit Price.


# Scenario 3 — Two-sample t-test

Business question: Is average Units sold different between Online and Offline sales?

H₀: Mean Online Units = Mean Offline Units
H₁: Mean Online Units ≠ Mean Offline Units

In [6]:
online_units = df[df["Channel"] == "Online"]["Units"]
offline_units = df[df["Channel"] == "Offline"]["Units"]

t_stat, p_value = stats.ttest_ind(
    online_units,
    offline_units,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject H0: Average Units sold differs significantly between Online and Offline sales.")
else:
    print("Fail to reject H0: No significant difference was found in Units sold between the two channels.")

T-statistic: nan
P-value: nan
Fail to reject H0: No significant difference was found in Units sold between the two channels.


# Confidence interval

For the Online vs Offline Unit Price comparison, you can calculate a 95% confidence interval for the difference in means.

In [9]:
import numpy as np

online_channels = ["Website", "App", "Marketplace"]

online = df[df["Channel"].isin(online_channels)]["Unit_Price"]
offline = df[df["Channel"] == "Offline"]["Unit_Price"]

mean_diff = online.mean() - offline.mean()

se = np.sqrt(
    online.var(ddof=1) / len(online) +
    offline.var(ddof=1) / len(offline)
)

df_welch = (
    (online.var(ddof=1)/len(online) + offline.var(ddof=1)/len(offline))**2
    /
    (
        (online.var(ddof=1)/len(online))**2/(len(online)-1)
        +
        (offline.var(ddof=1)/len(offline))**2/(len(offline)-1)
    )
)

critical = stats.t.ppf(0.975, df_welch)

lower = mean_diff - critical * se
upper = mean_diff + critical * se

print("Mean Difference:", mean_diff)
print("95% Confidence Interval:", (lower, upper))

Mean Difference: -46.3211705591857
95% Confidence Interval: (np.float64(-238.6782413506878), np.float64(146.0359002323164))


# about p value

Result	Interpretation
p < 0.05	Reject H₀
p ≥ 0.05	Fail to reject H₀

A p-value is not the probability that H₀ is true. It measures how unusual the observed result would be if the null hypothesis were true.

# Type I and Type II errors

Type I Error

A Type I error occurs when we reject a true null hypothesis.

Business example:
A company concludes that Online and Offline customers have different average prices when actually there is no real difference.

Consequence:
The company might unnecessarily change pricing, promotions, or channel strategy.

Type II Error

A Type II error occurs when we fail to reject a false null hypothesis.

Business example:
A company concludes that Online and Offline prices are not significantly different when a real difference exists.

Consequence:
The company may miss an opportunity to optimise pricing or marketing strategy.